# Random Forest: Additional Experiments

We investigate whether using more detailed Moodle activity features and longer activity periods improves the prediction of students' final numerical grades.

We first compare different feature sets using the first 8 weeks of activity. Then, we compare different prediction periods using the selected features.

The original Random Forest model is kept as a reference.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, LeaveOneGroupOut
from sklearn.metrics import mean_absolute_error

RANDOM_STATE = 42

## 1. Load the data

We use the same training and testing datasets as the original models. The additional features will be calculated from Moodle event logs.

In [3]:
train = pd.read_csv("shared_train.csv")
test = pd.read_csv("shared_test.csv")

FEATURES = [
    "total_events",
    "active_days",
    "course_module_views",
    "course_views"
]

TARGET = "course_numeric_grade"

print("Training records:", len(train))
print("Testing records:", len(test))

print("Training years:", sorted(train["year"].unique()))
print("Testing years:", sorted(test["year"].unique()))

display(train.head())

Training records: 551
Testing records: 279
Training years: [np.int64(2019), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Testing years: [np.int64(2025), np.int64(2026)]


,year,actor_id,course_numeric_grade,total_events,active_days,course_module_views,course_views
0,2019,S027cf9,72.10,132,18,20,34
1,2019,S032255,67.70,48,6,18,8
2,2019,S03a8b8,107.02,221,30,39,75
3,2019,S04cf7f,85.85,161,20,26,38
4,2019,S05f285,37.15,456,32,76,177


In [4]:
print("Original predictor columns:")
print(FEATURES)

print("\nMissing values in the original predictors:")
display(train[FEATURES].isna().sum())

Original predictor columns:
['total_events', 'active_days', 'course_module_views', 'course_views']

Missing values in the original predictors:


total_events           0
active_days            0
course_module_views    0
course_views           0
dtype: int64

## 2. Prepare additional features

We create additional predictors from Moodle activity during the first 8 weeks of each course. We include specific activity types and measures of regularity to see whether they provide more useful information than general event counts.

In [6]:
from pathlib import Path

DATA_DIR = Path("tcs_dataset/data")

events_list = []

for year in range(2019, 2027):
    df = pd.read_parquet(
        DATA_DIR / str(year) / "events.parquet",
        columns=["actor_id", "event_time", "event_name"]
    )

    df = df[df["actor_id"].str.startswith("S", na=False)].copy()
    df["year"] = year

    events_list.append(df)

events = pd.concat(events_list, ignore_index=True)

print("Student events:", len(events))
display(events.head())


Student events: 733090


,actor_id,event_time,event_name,year
0,Sd73b43,2019-02-11 12:20:00+02:00,Course viewed,2019
1,Sd73b43,2019-02-11 12:21:00+02:00,Grade user report viewed,2019
2,Sd2e352,2019-02-11 12:21:00+02:00,Course viewed,2019
3,Sd2e352,2019-02-11 12:21:00+02:00,Grade user report viewed,2019
4,Sd2e352,2019-02-11 12:21:00+02:00,Course viewed,2019


In [11]:
course_runs = pd.read_csv(DATA_DIR / "course_runs.csv")

course_runs["year"] = course_runs["course_run_id"].str[-4:].astype(int)
course_runs["course_start_date"] = pd.to_datetime(
    course_runs["course_start_date"]
)

start_dates = course_runs.set_index("year")["course_start_date"]

events["course_start"] = pd.to_datetime(
    events["year"].map(start_dates),
    utc=True
).dt.tz_convert("Europe/Tallinn")

events["cutoff"] = events["course_start"] + pd.Timedelta(weeks=8)

early = events[
    (events["event_time"] >= events["course_start"]) &
    (events["event_time"] < events["cutoff"])
].copy()

print("Events within the first 8 weeks:", len(early))

Events within the first 8 weeks: 218194


### 2.1 Selected activity features

We count quiz attempts, assignment submissions and recording views for each student. We also measure how regularly students were active during the first 8 weeks.

In [12]:
event_features = {
    "quiz_attempts_started": "Quiz attempt started",
    "quiz_attempts_submitted": "Quiz attempt submitted",
    "assignment_submissions": "A submission has been submitted.",
    "recording_views": "Recording viewed"
}

selected_features = pd.DataFrame(
    index=pd.MultiIndex.from_frame(early[["year", "actor_id"]].drop_duplicates())
)

for feature, event_name in event_features.items():
    counts = (
        early[early["event_name"] == event_name]
        .groupby(["year", "actor_id"])
        .size()
    )
    selected_features[feature] = counts

selected_features = selected_features.fillna(0).reset_index()

display(selected_features.head())


,year,actor_id,quiz_attempts_started,quiz_attempts_submitted,assignment_submissions,recording_views
0,2019,Sd73b43,2.0,2.0,5.0,0.0
1,2019,Sd2e352,2.0,2.0,4.0,0.0
2,2019,S968095,2.0,2.0,1.0,0.0
3,2019,S4814c5,2.0,2.0,4.0,0.0
4,2019,S57d0df,2.0,2.0,1.0,0.0


In [13]:
# Count distinct weeks with Moodle activity
early["week"] = (
    (early["event_time"] - early["course_start"]).dt.total_seconds()
    // (7 * 24 * 60 * 60)
).astype(int)

active_weeks = (
    early.groupby(["year", "actor_id"])["week"]
    .nunique()
    .rename("active_weeks")
    .reset_index()
)

selected_features = selected_features.merge(
    active_weeks,
    on=["year", "actor_id"],
    how="left"
)

display(selected_features.head())

,year,actor_id,quiz_attempts_started,quiz_attempts_submitted,assignment_submissions,recording_views,active_weeks
0,2019,Sd73b43,2.0,2.0,5.0,0.0,8
1,2019,Sd2e352,2.0,2.0,4.0,0.0,8
2,2019,S968095,2.0,2.0,1.0,0.0,8
3,2019,S4814c5,2.0,2.0,4.0,0.0,8
4,2019,S57d0df,2.0,2.0,1.0,0.0,8


## 3. Compare feature sets

We compare the original four Moodle activity features with an extended set that includes specific activity types and active weeks. Both use the first 8 weeks of activity and the same training data.

In [21]:
train_extended = train.merge(
    selected_features,
    on=["year", "actor_id"],
    how="left"
)

test_extended = test.merge(
    selected_features,
    on=["year", "actor_id"],
    how="left"
)

# Students without these activities receive zero counts
new_columns = [
    "quiz_attempts_started",
    "quiz_attempts_submitted",
    "assignment_submissions",
    "recording_views",
    "active_weeks"
]

train_extended[new_columns] = train_extended[new_columns].fillna(0)
test_extended[new_columns] = test_extended[new_columns].fillna(0)

print("Training records:", len(train_extended))
print("Testing records:", len(test_extended))

display(train_extended.head())

Training records: 551
Testing records: 279


,year,actor_id,course_numeric_grade,total_events,active_days,course_module_views,course_views,quiz_attempts_started,quiz_attempts_submitted,assignment_submissions,recording_views,active_weeks
0,2019,S027cf9,72.10,132,18,20,34,2.0,2.0,3.0,0.0,6
1,2019,S032255,67.70,48,6,18,8,2.0,1.0,0.0,0.0,5
2,2019,S03a8b8,107.02,221,30,39,75,2.0,2.0,3.0,0.0,8
3,2019,S04cf7f,85.85,161,20,26,38,2.0,2.0,6.0,0.0,8
4,2019,S05f285,37.15,456,32,76,177,2.0,2.0,5.0,0.0,7


In [16]:
# Original predictors
features_A = FEATURES

# Original predictors + selected activity features
features_B = FEATURES + new_columns

print("Feature set A:", features_A)
print("\nFeature set B:", features_B)

Feature set A: ['total_events', 'active_days', 'course_module_views', 'course_views']

Feature set B: ['total_events', 'active_days', 'course_module_views', 'course_views', 'quiz_attempts_started', 'quiz_attempts_submitted', 'assignment_submissions', 'recording_views', 'active_weeks']


### 3.1 All event-type features

We create a separate predictor for every Moodle event type recorded during the first 8 weeks.

Unlike Set B, which includes only selected activities, Set C includes all student event types. This allows us to test whether more detailed activity information improves grade predictions.

We use event counts rather than grades or assessment scores.

In [17]:
all_event_features = pd.crosstab(
    [early["year"], early["actor_id"]],
    early["event_name"]
)

all_event_features = all_event_features.add_prefix("event_")

all_event_features = all_event_features.reset_index()

print("Number of event types:", all_event_features.shape[1] - 2)
display(all_event_features.head())

Number of event types: 42


event_name,year,actor_id,event_A file has been uploaded.,event_A submission has been submitted.,event_An online text has been uploaded.,event_BigBlueButton activity viewed,event_BigBlueButton session created,event_BigBlueButton session joined,event_BigBlueButtonBN activity viewed,event_BigBlueButtonBN meeting created,...,event_Remove submission confirmation viewed.,event_Some content has been posted.,event_Submission created.,event_Submission form viewed.,event_Submission updated.,event_The status of the submission has been updated.,event_The status of the submission has been viewed.,event_User graded,event_User list viewed,event_User profile viewed
0,2019,S027cf9,3,3,3,0,0,0,0,0,...,0,0,6,3,0,0,36,4,0,1
1,2019,S032255,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,14,1,0,0
2,2019,S03a8b8,3,3,3,0,0,0,0,0,...,0,0,6,3,0,0,41,4,0,0
3,2019,S04cf7f,6,6,6,0,0,0,0,0,...,0,0,6,7,6,0,34,4,0,0
4,2019,S05f285,5,5,5,0,0,0,0,0,...,0,0,10,8,0,0,75,4,3,2


### 3.2 Merge event features

We add the event-type counts to the original student datasets. Students without recorded activity are kept, and their missing event counts are replaced with zeros.

In [18]:
train_all = train.merge(
    all_event_features,
    on=["year", "actor_id"],
    how="left"
)

test_all = test.merge(
    all_event_features,
    on=["year", "actor_id"],
    how="left"
)

event_columns = [
    col for col in all_event_features.columns
    if col.startswith("event_")
]

train_all[event_columns] = train_all[event_columns].fillna(0)
test_all[event_columns] = test_all[event_columns].fillna(0)

print("Training records:", len(train_all))
print("Testing records:", len(test_all))

Training records: 551
Testing records: 279


In [20]:
event_columns = [
    col for col in event_columns
    if train_all[col].sum() > 0
]

# Original predictors + all event-type counts
features_C = FEATURES + event_columns

print("Feature Set A:", len(features_A), "predictors")
print("Feature Set B:", len(features_B), "predictors")
print("Feature Set C:", len(features_C), "predictors")

Feature Set A: 4 predictors
Feature Set B: 9 predictors
Feature Set C: 39 predictors


In [22]:
print("Missing values in Set C:")
print(train_all[features_C].isna().sum().sum())

print("\nDuplicate student-year records:")
print(train_all.duplicated(["year", "actor_id"]).sum())

print("\nNumber of predictors:", len(features_C))

Missing values in Set C:
0

Duplicate student-year records:
0

Number of predictors: 39


### 3.3 Random Forest comparison

We compare the three feature sets using Leave-One-Year-Out cross-validation.

All models use the same Random Forest settings and the same 8-week activity period. We compare their mean absolute errors (MAE), where lower values indicate better predictions.

In [23]:
feature_sets = {
    "A: Original": (train, features_A),
    "B: Selected": (train_extended, features_B),
    "C: All events": (train_all, features_C)
}

logo = LeaveOneGroupOut()
comparison = []

for name, (dataset, features) in feature_sets.items():

    X = dataset[features]
    y = dataset[TARGET]
    groups = dataset["year"]

    model = RandomForestRegressor(
        n_estimators=100,
        max_depth=1,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    scores = []

    for train_idx, val_idx in logo.split(X, y, groups):
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        predictions = model.predict(X.iloc[val_idx])

        mae = mean_absolute_error(
            y.iloc[val_idx],
            predictions
        )
        scores.append(mae)

    comparison.append({
        "Feature set": name,
        "Predictors": len(features),
        "CV MAE": np.mean(scores)
    })

comparison_df = pd.DataFrame(comparison)

display(comparison_df.round(2))

,Feature set,Predictors,CV MAE
0,A: Original,4,17.87
1,B: Selected,9,20.07
2,C: All events,39,19.81


### 3.4 Performance across years

We compare the prediction errors for each validation year separately. This helps us identify whether additional features perform consistently or whether certain course years are more difficult to predict.

In [24]:
# Compare Random Forest errors for each validation year
yearly_results = []

for name, (dataset, features) in feature_sets.items():

    X = dataset[features]
    y = dataset[TARGET]
    groups = dataset["year"]

    for train_idx, val_idx in logo.split(X, y, groups):

        model = RandomForestRegressor(
            n_estimators=100,
            max_depth=1,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        predictions = model.predict(X.iloc[val_idx])

        yearly_results.append({
            "Year": groups.iloc[val_idx].iloc[0],
            "Feature set": name,
            "MAE": mean_absolute_error(y.iloc[val_idx], predictions)
        })

yearly_df = pd.DataFrame(yearly_results)

display(
    yearly_df.pivot(
        index="Year",
        columns="Feature set",
        values="MAE"
    ).round(2)
)

Feature set,A: Original,B: Selected,C: All events
Year,,,
2019,17.52,28.82,27.56
2021,19.31,19.27,19.19
2022,15.90,15.84,15.81
2023,20.24,20.07,20.14
2024,16.39,16.37,16.33


### Interpretation

The additional predictors increased the overall cross-validation MAE, mainly because of the results for 2019. For this year, the original feature set achieved an MAE of 17.52, while the selected features and all event types produced much higher errors of 28.82 and 27.56.

For 2021–2024, the differences between the feature sets were small. The additional event features slightly improved predictions in most years, but the improvements were limited.

This suggests that detailed Moodle activity features may not generalise equally well across different course years. One possible explanation is that teaching arrangements and Moodle usage were different in 2019.

We will investigate whether including 2019 in the training data affects prediction accuracy for later years.


### 3.5 Effect of including 2019 in training

The previous results suggest that 2019 differs from the later course years. We compare models trained with and without 2019 to investigate whether including this year improves predictions for the 2021–2024 cohorts.

Both approaches are evaluated on the same validation years using Leave-One-Year-Out cross-validation.

In [26]:
year_experiment = []

for name, (dataset, features) in feature_sets.items():

    for val_year in [2021, 2022, 2023, 2024]:

        for include_2019 in [True, False]:

            train_mask = dataset["year"] != val_year

            if not include_2019:
                train_mask &= dataset["year"] != 2019

            val_mask = dataset["year"] == val_year

            model = RandomForestRegressor(
                n_estimators=100,
                max_depth=1,
                random_state=RANDOM_STATE,
                n_jobs=-1
            )

            model.fit(
                dataset.loc[train_mask, features],
                dataset.loc[train_mask, TARGET]
            )

            predictions = model.predict(
                dataset.loc[val_mask, features]
            )

            mae = mean_absolute_error(
                dataset.loc[val_mask, TARGET],
                predictions
            )

            year_experiment.append({
                "Feature set": name,
                "Validation year": val_year,
                "Training": "With 2019" if include_2019 else "Without 2019",
                "MAE": mae
            })

year_experiment_df = pd.DataFrame(year_experiment)

display(
    year_experiment_df.pivot_table(
        index=["Feature set", "Validation year"],
        columns="Training",
        values="MAE"
    ).round(2)
)

Training                       With 2019  Without 2019
Feature set   Validation year                         
A: Original   2021                 19.31         19.27
              2022                 15.90         16.36
              2023                 20.24         20.55
              2024                 16.39         16.76
B: Selected   2021                 19.27         18.08
              2022                 15.84         16.01
              2023                 20.07         20.94
              2024                 16.37         15.95
C: All events 2021                 19.19         18.05
              2022                 15.81         15.94
              2023                 20.14         20.98
              2024                 16.33         15.90

### Interpretation

Removing 2019 from training did not consistently improve prediction accuracy. The original feature set performed slightly better when 2019 was included, while the extended feature sets improved only slightly when it was excluded.

This suggests that although Moodle activity patterns may have been different in 2019, its data can still be useful for predicting grades in later years. Since the differences were small, we decided to keep 2019 in the training dataset.

### 3.6 Differences in event types across years

We examine whether Moodle event types are recorded consistently across course years. Some activities may be missing or more common in certain years because of differences in teaching arrangements or Moodle functionality.

These differences could affect how well the models generalise between cohorts.

In [27]:
# Count each event type in each year
events_by_year = pd.crosstab(
    early["event_name"],
    early["year"]
)

# Count how many years each event type appears in
events_by_year["years_present"] = (
    events_by_year > 0
).sum(axis=1)

# Show events that do not appear in every year
inconsistent_events = events_by_year[
    events_by_year["years_present"] < early["year"].nunique()
].sort_values("years_present")

print("Event types present in all years:",
      (events_by_year["years_present"] == early["year"].nunique()).sum())

print("Event types missing from at least one year:",
      len(inconsistent_events))

display(inconsistent_events)

Event types present in all years: 14
Event types missing from at least one year: 28


year,2019,2020,2021,2022,2023,2024,2025,2026,years_present
event_name,,,,,,,,,
BigBlueButtonBN activity viewed,0,496,0,0,0,0,0,0,1
BigBlueButtonBN meeting created,0,20,0,0,0,0,0,0,1
Course searched,0,0,0,1,0,0,0,0,1
BigBlueButtonBN meeting joined,0,214,0,0,0,0,0,0,1
Feedback viewed,0,178,0,0,0,0,0,0,1
Remove submission confirmation viewed.,0,4,0,0,0,0,0,0,1
Quiz submitted automatically,0,0,0,0,0,0,0,2,1
Post updated,0,0,2,0,0,0,0,0,1
The status of the submission has been updated.,0,4,0,0,0,0,0,0,1


### Interpretation

Only 14 of the 42 Moodle event types appeared in all eight years, while 28 were absent in at least one year.

Some differences were substantial. For example, assignment submission events were common in 2019 and 2020 but mostly absent in later years, while `Quiz attempt updated` appeared only from 2024 onward. BigBlueButton and recording activities also varied between years.

These differences suggest that Moodle activity types were not entirely consistent across course years. Therefore, detailed event counts may reflect differences in Moodle tools and teaching arrangements rather than only differences in student behaviour.

This could help explain why adding more event-type predictors did not improve the Random Forest's overall cross-validation performance.

### 3.7 Consistent event-type features

Since many Moodle event types are missing in some years, we test whether using only consistently recorded event types improves predictions.

We select event types appearing in every training year (2019 and 2021–2024). This selection does not use information from the test years.


In [28]:
# Identify event types present in every training year
training_years = sorted(train["year"].unique())

training_events = early[
    early["year"].isin(training_years)
]

event_presence = pd.crosstab(
    training_events["event_name"],
    training_events["year"]
)

consistent_events = event_presence.index[
    (event_presence > 0).all(axis=1)
].tolist()

print("Consistent event types:", len(consistent_events))
display(pd.DataFrame({"Event type": consistent_events}))

Consistent event types: 14


,Event type
0,Course module instance list viewed
1,Course module viewed
2,Course user report viewed
3,Course viewed
4,Grade overview report viewed
5,Grade user report viewed
6,Quiz attempt reviewed
7,Quiz attempt started
8,Quiz attempt submitted
9,Quiz attempt summary viewed


### 3.7 Feature Set D: Consistent event types

We create a fourth feature set using only event types that appear in every training year (2019 and 2021–2024).

This allows us to test whether consistently recorded Moodle activities improve prediction accuracy compared to using all event types.

We exclude `User graded` because its meaning was unclear, even though it appeared under student IDs.

In [31]:
consistent_events_D = [
    event for event in consistent_events
    if event != "User graded"
]

consistent_columns = [
    "event_" + event
    for event in consistent_events_D
]

# Original predictors + consistent event types
features_D = FEATURES + consistent_columns

print("Number of predictors in Set D:", len(features_D))
display(pd.DataFrame({"Event type": consistent_events_D}))

Number of predictors in Set D: 17


,Event type
0,Course module instance list viewed
1,Course module viewed
2,Course user report viewed
3,Course viewed
4,Grade overview report viewed
5,Grade user report viewed
6,Quiz attempt reviewed
7,Quiz attempt started
8,Quiz attempt submitted
9,Quiz attempt summary viewed


### 3.8 Comparison of four feature sets

We compare all four feature sets using the same Random Forest settings and Leave-One-Year-Out cross-validation.

All models use the first 8 weeks of Moodle activity. This allows us to investigate whether the choice of predictors affects prediction accuracy.

In [32]:
# Add Feature Set D
feature_sets["D: Consistent"] = (train_all, features_D)

comparison = []

for name, (dataset, features) in feature_sets.items():

    X = dataset[features]
    y = dataset[TARGET]
    groups = dataset["year"]

    scores = []

    for train_idx, val_idx in logo.split(X, y, groups):

        model = RandomForestRegressor(
            n_estimators=100,
            max_depth=1,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        predictions = model.predict(X.iloc[val_idx])

        scores.append(
            mean_absolute_error(y.iloc[val_idx], predictions)
        )

    comparison.append({
        "Feature set": name,
        "Predictors": len(features),
        "CV MAE": np.mean(scores)
    })

comparison_df = pd.DataFrame(comparison)

display(comparison_df.round(2))

,Feature set,Predictors,CV MAE
0,A: Original,4,17.87
1,B: Selected,9,20.07
2,C: All events,39,19.81
3,D: Consistent,17,19.81


### Interpretation

The original feature set achieved the lowest cross-validation MAE (17.87). Adding selected activities, all event types, or only consistently recorded event types did not improve performance with the current Random Forest settings.

Sets C and D achieved the same MAE of 19.81, suggesting that restricting the predictors to consistently recorded events did not help in this experiment.

However, all four models used a maximum tree depth of 1. Since these settings were originally selected for Set A, we will optimise the hyperparameters separately for each feature set before drawing final conclusions.

## 4. Hyperparameter optimisation

We use GridSearchCV with Leave-One-Year-Out cross-validation to find suitable Random Forest settings for each feature set.

We tune tree depth, minimum leaf size and number of trees. The same parameter grid is used for all four feature sets to make the comparison fair.

Models are evaluated using mean absolute error (MAE), where lower values indicate better predictions.


In [33]:
# Hyperparameters to test
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [1, 2, 3, 5, None],
    "min_samples_leaf": [1, 5, 10]
}

logo = LeaveOneGroupOut()

print("Parameter combinations:",
      np.prod([len(values) for values in param_grid.values()]))


Parameter combinations: 30


### 4.1 Tuning results

We optimise each feature set independently using the same cross-validation procedure. We compare the best parameter combinations and their validation errors to determine whether more detailed activity predictors benefit from different model settings.

In [34]:
tuning_results = []
best_models = {}

for name, (dataset, features) in feature_sets.items():

    print("Tuning:", name)

    X = dataset[features]
    y = dataset[TARGET]
    groups = dataset["year"]

    grid = GridSearchCV(
        estimator=RandomForestRegressor(
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        param_grid=param_grid,
        scoring="neg_mean_absolute_error",
        cv=logo,
        n_jobs=-1
    )

    grid.fit(X, y, groups=groups)

    best_models[name] = grid.best_estimator_

    tuning_results.append({
        "Feature set": name,
        "Predictors": len(features),
        "Best CV MAE": -grid.best_score_,
        "Best parameters": grid.best_params_
    })

tuning_df = pd.DataFrame(tuning_results)

display(tuning_df.round(2))

Tuning: A: Original
Tuning: B: Selected
Tuning: C: All events
Tuning: D: Consistent


,Feature set,Predictors,Best CV MAE,Best parameters
0,A: Original,4,17.87,"{'max_depth': 1, 'min_samples_leaf': 1, 'n_est..."
1,B: Selected,9,20.00,"{'max_depth': 1, 'min_samples_leaf': 10, 'n_es..."
2,C: All events,39,19.80,"{'max_depth': 1, 'min_samples_leaf': 10, 'n_es..."
3,D: Consistent,17,19.80,"{'max_depth': 1, 'min_samples_leaf': 10, 'n_es..."


### Interpretation

Hyperparameter tuning did not significantly improve the performance of the extended feature sets. All four models achieved their best results with a maximum tree depth of 1.

The original four predictors still produced the lowest cross-validation MAE (17.87), while the extended feature sets achieved MAEs between 19.80 and 20.00.

These results suggest that adding more detailed Moodle event counts does not necessarily improve grade prediction. One possible explanation is that specific Moodle activities vary across course years, making them less reliable for predicting results in unseen cohorts.

Overall, the simpler model performed better and generalised more consistently across years.


## 5. Comparing prediction periods

We investigate whether using more weeks of Moodle activity improves final grade predictions.

We compare activity from the first 8, 10 and 12 weeks while keeping the original four predictors. This allows us to examine whether additional activity over time provides useful information.

We use Leave-One-Year-Out cross-validation on the training years to compare the prediction periods.


In [35]:
# Prediction periods to compare
WEEKS = [8, 10, 12]

# Load official course start dates
course_runs = pd.read_csv(DATA_DIR / "course_runs.csv")
course_runs["year"] = course_runs["course_run_id"].str[-4:].astype(int)
course_runs["course_start_date"] = pd.to_datetime(
    course_runs["course_start_date"], utc=True
)

start_dates = course_runs.set_index("year")["course_start_date"]

# Use the same UTC timestamp handling as the original shared notebook
events["event_time"] = pd.to_datetime(events["event_time"], utc=True)

def create_week_features(weeks):
    cutoff = events["year"].map(start_dates) + pd.Timedelta(weeks=weeks)

    period_events = events[events["event_time"] < cutoff].copy()

    period_events["date"] = (
        period_events["event_time"]
        .dt.tz_convert("Europe/Tallinn")
        .dt.date
    )

    period_events["is_module_view"] = (
        period_events["event_name"] == "Course module viewed"
    ).astype(int)

    period_events["is_course_view"] = (
        period_events["event_name"] == "Course viewed"
    ).astype(int)

    features = period_events.groupby(["year", "actor_id"]).agg(
        total_events=("event_name", "size"),
        active_days=("date", "nunique"),
        course_module_views=("is_module_view", "sum"),
        course_views=("is_course_view", "sum")
    ).reset_index()

    return features

In [36]:
# Recreate the original 8-week features
features_8 = create_week_features(8)

# Merge with the original student records
check_8 = train[["year", "actor_id"] + FEATURES].merge(
    features_8,
    on=["year", "actor_id"],
    how="left",
    suffixes=("_original", "_new")
)

# Students without activity receive zeros
for feature in FEATURES:
    check_8[feature + "_new"] = check_8[feature + "_new"].fillna(0)

# Compare the original and recreated values
for feature in FEATURES:
    differences = (
        check_8[feature + "_original"] !=
        check_8[feature + "_new"]
    ).sum()

    print(feature, "differences:", differences)

total_events differences: 0
active_days differences: 0
course_module_views differences: 0
course_views differences: 0


### 5.1 Cross-validation results

We calculate the original four predictors using 8, 10 and 12 weeks of Moodle activity.

We keep the same Random Forest settings and use Leave-One-Year-Out cross-validation to compare prediction accuracy. Lower MAE indicates better performance.

In [38]:
# Compare different prediction periods
week_results = []
week_datasets = {}

for weeks in WEEKS:

    # Calculate activity features for this period
    period_features = create_week_features(weeks)

    # Keep the same students and grades as the original training data
    dataset = train[["year", "actor_id", TARGET]].merge(
        period_features,
        on=["year", "actor_id"],
        how="left"
    )

    dataset[FEATURES] = dataset[FEATURES].fillna(0)
    week_datasets[weeks] = dataset

    X = dataset[FEATURES]
    y = dataset[TARGET]
    groups = dataset["year"]

    scores = []

    for train_idx, val_idx in logo.split(X, y, groups):

        model = RandomForestRegressor(
            n_estimators=100,
            max_depth=1,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        predictions = model.predict(X.iloc[val_idx])

        scores.append(
            mean_absolute_error(y.iloc[val_idx], predictions)
        )

    week_results.append({
        "Weeks": weeks,
        "CV MAE": np.mean(scores)
    })

week_comparison = pd.DataFrame(week_results)

display(week_comparison.round(2))

,Weeks,CV MAE
0,8,17.87
1,10,18.09
2,12,17.78


### Interpretation

Increasing the activity period did not substantially improve prediction accuracy. The 12-week model achieved the lowest MAE (17.78), but the improvement over 8 weeks (17.87) was very small. The 10-week model performed slightly worse (18.09).

These results suggest that simply including more Moodle activity does not provide much additional predictive information for the current Random Forest model.

Since the differences are small, the 8-week model remains a reasonable choice because it allows predictions to be made earlier.


## 6. Activity patterns over time

The previous experiments showed that additional event types and longer activity periods did not substantially improve prediction accuracy.

We now investigate whether activity patterns during the first 8 weeks contain useful information. Instead of only counting total activity, we measure how activity changes and how consistently students use Moodle.


In [39]:
# Calculate the course week for each event
weekly_events = events.copy()

weekly_events["course_start"] = weekly_events["year"].map(start_dates)

weekly_events["week"] = (
    (weekly_events["event_time"] - weekly_events["course_start"])
    .dt.total_seconds() // (7 * 24 * 60 * 60)
).astype(int) + 1

# Keep only the first 8 weeks
weekly_events = weekly_events[
    weekly_events["week"].between(1, 8)
].copy()

# Count events per student per week
weekly_counts = (
    weekly_events.groupby(["year", "actor_id", "week"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=range(1, 9), fill_value=0)
)

display(weekly_counts.head())


week            1   2   3   4   5    6   7   8
year actor_id                                 
2019 S027cf9    0   0  19  29  23   18  16  27
     S032255    0   0   0   5  14    8   1  20
     S03a8b8   21   9  16   8  37   41  65  24
     S04cf7f    1   3   1  15  45   38  39  19
     S05f285    0  13   5  58  76  144  91  69

### 6.1 Create activity-pattern predictors

We calculate the amount of activity in the first and second halves of the 8-week period, the difference between them, the number of active weeks and the longest period without activity.


In [41]:
# Divide activity into two periods
pattern_features = pd.DataFrame(index=weekly_counts.index)

pattern_features["first_half_events"] = weekly_counts[[1, 2, 3, 4]].sum(axis=1)
pattern_features["second_half_events"] = weekly_counts[[5, 6, 7, 8]].sum(axis=1)

# Change in activity between the two periods
pattern_features["activity_change"] = (
    pattern_features["second_half_events"]
    - pattern_features["first_half_events"]
)

# Number of weeks with any activity
pattern_features["active_weeks"] = (weekly_counts > 0).sum(axis=1)

# Longest sequence of inactive weeks
def longest_inactive_streak(row):
    longest = 0
    current = 0

    for count in row:
        if count == 0:
            current += 1
            longest = max(longest, current)
        else:
            current = 0

    return longest

pattern_features["max_inactive_streak"] = weekly_counts.apply(
    longest_inactive_streak,
    axis=1
)

pattern_features = pattern_features.reset_index()

display(pattern_features.head())


,year,actor_id,first_half_events,second_half_events,activity_change,active_weeks,max_inactive_streak
0,2019,S027cf9,48,84,36,6,2
1,2019,S032255,5,43,38,5,3
2,2019,S03a8b8,54,167,113,8,0
3,2019,S04cf7f,20,141,121,8,0
4,2019,S05f285,76,380,304,7,1


### 6.2 Feature Set E: Activity patterns

We combine the original four predictors with five activity-pattern features.

This allows us to compare whether the timing and consistency of Moodle activity provide more useful information than overall activity counts alone.


In [42]:
# Define activity-pattern predictors
pattern_columns = [
    "first_half_events",
    "second_half_events",
    "activity_change",
    "active_weeks",
    "max_inactive_streak"
]

# Add patterns to the original training dataset
train_patterns = train.merge(
    pattern_features,
    on=["year", "actor_id"],
    how="left"
)

# Fill missing values for students with no activity
train_patterns[pattern_columns] = train_patterns[pattern_columns].fillna(0)

# Students with no activity were inactive for all 8 weeks
train_patterns.loc[
    train_patterns["active_weeks"] == 0,
    "max_inactive_streak"
] = 8

# Feature Set E
features_E = FEATURES + pattern_columns

print("Training records:", len(train_patterns))
print("Number of predictors:", len(features_E))
print("Missing values:", train_patterns[features_E].isna().sum().sum())


Training records: 551
Number of predictors: 9
Missing values: 0


### 6.3 Activity-pattern comparison

We compare the original predictors (Set A) with the extended activity-pattern predictors (Set E).

Both models use the first 8 weeks of Moodle activity and the same Random Forest settings. We evaluate them using Leave-One-Year-Out cross-validation.


In [44]:
# Compare original features with activity patterns
pattern_comparison = []

pattern_feature_sets = {
    "A: Original": (train, features_A),
    "E: Activity patterns": (train_patterns, features_E)
}

for name, (dataset, features) in pattern_feature_sets.items():

    X = dataset[features]
    y = dataset[TARGET]
    groups = dataset["year"]

    scores = []

    for train_idx, val_idx in logo.split(X, y, groups):

        model = RandomForestRegressor(
            n_estimators=100,
            max_depth=1,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        predictions = model.predict(X.iloc[val_idx])

        scores.append(mean_absolute_error(y.iloc[val_idx], predictions))

    pattern_comparison.append({
        "Feature set": name,
        "Predictors": len(features),
        "CV MAE": np.mean(scores)
    })

pattern_comparison_df = pd.DataFrame(pattern_comparison)

display(pattern_comparison_df.round(2))


,Feature set,Predictors,CV MAE
0,A: Original,4,17.87
1,E: Activity patterns,9,17.86


### Interpretation

Adding activity-pattern features slightly reduced the cross-validation MAE from 17.87 to 17.86. However, the improvement of 0.01 points is negligible and does not provide convincing evidence that these features improve prediction accuracy.

Overall, the experiments showed that adding more event types, measuring activity patterns, optimising hyperparameters and extending the prediction period did not substantially improve the Random Forest's performance.

## 7. Summary. Overall results and future work

### Model comparison

We initially trained three regression models to predict students' final numerical grades using Moodle activity from the first 8 weeks: Ridge Regression, Decision Tree and Random Forest.

All three models were trained on data from 2019 and 2021–2024 and evaluated on the unseen 2025 and 2026 cohorts.

| Model | 2025 MAE | 2026 MAE |
|---|---:|---:|
| Baseline | 20.21 | 21.02 |
| Ridge Regression | **18.34** | **19.53** |
| Decision Tree | 19.65 | 21.64 |
| Random Forest | 18.78 | 20.77 |

Ridge Regression achieved the lowest MAE in both test years, making it the best-performing model in our initial comparison. Random Forest also performed better than the baseline in both years, although its improvement was relatively small, especially in 2026. Decision Tree performed worse than the baseline in 2026.

Overall, these results suggest that early Moodle activity contains some information about students' final grades, but the prediction errors remain relatively high.

### Additional Random Forest experiments

To investigate whether prediction accuracy could be improved, we conducted several additional experiments using Random Forest.

First, we compared different feature sets using the same 8-week activity period. We tested the original four predictors, selected activity types, all available event types and only event types consistently recorded across training years.

The original four predictors achieved the lowest cross-validation MAE of 17.87. Adding more detailed event counts did not improve performance, even after optimising the Random Forest hyperparameters.

We also found that Moodle event types varied considerably across years. Only 14 of the 42 event types appeared in every year, suggesting that differences in Moodle tools and teaching arrangements may make detailed event features less reliable across cohorts.

We investigated whether excluding 2019 from training would improve performance, since its activity patterns differed from later years. However, removing 2019 did not consistently reduce prediction errors, so we decided to keep it.

Next, we compared activity periods of 8, 10 and 12 weeks. The 12-week model achieved a slightly lower cross-validation MAE (17.78) than the 8-week model (17.87), but the improvement was negligible. Therefore, using more weeks did not provide a substantial benefit.

Finally, we tested additional features describing activity patterns, including changes in activity between the first and second halves of the 8-week period, active weeks and periods of inactivity. This reduced the cross-validation MAE only slightly, from 17.87 to 17.86.

Overall, none of these experiments produced a meaningful improvement over the original Random Forest model.

### Main findings

Our results suggest that simply increasing the number of Moodle activity predictors or extending the observation period does not necessarily improve final grade predictions.

One possible explanation is that the current features mainly describe how frequently students interact with Moodle, rather than how well they understand the course material. Students with similar activity levels can still achieve very different final grades.

The differences in Moodle usage and recorded event types across course years may also limit how well models generalise to new cohorts.

The original four-feature model therefore remains a reasonable choice because it provides comparable accuracy with fewer predictors and allows predictions to be made earlier in the semester.

### Future work

Our next step is to investigate whether more meaningful predictors can improve prediction accuracy.

Instead of focusing only on Moodle activity counts, we plan to explore students' academic progress using information from the original dataset.

Possible improvements include:

1. **Early homework performance:** Use completed homework results to investigate whether earlier academic performance is a stronger predictor of final grades than Moodle activity alone.

2. **Assessment-based prediction:** Investigate whether homework and midterm results can help predict students' later exam performance.

3. **Deadline-related activity:** Use course calendars and event timestamps to examine students' activity around homework deadlines, rather than only counting total Moodle events.

4. **Combined predictors:** Compare models using Moodle activity alone, early assessment results alone, and both together to determine whether Moodle activity provides additional predictive information.

5. **Alternative prediction targets:** Consider predicting final exam performance or whether students are at risk of failing, instead of predicting only the exact final numerical grade.

Before using assessment results as predictors, we will need to verify when they became available to students. This is important to avoid using information that would not have been known at the intended prediction point.

### Conclusion

The initial experiments showed that Ridge Regression performed best among the three models, while the additional Random Forest experiments demonstrated that more detailed activity features and longer observation periods did not substantially improve accuracy.

These results suggest that the main limitation may be the information contained in the predictors rather than the complexity of the models.

Our future work will therefore focus on finding more meaningful indicators of student progress, especially homework performance, assessment results and deadline-related behaviour. We aim to determine whether these features can provide more accurate and useful predictions of students' academic outcomes.
